# Day 11 – Fine‑tuning with LoRA – Extensive Solutions

This notebook provides complete solutions for all exercises, including detailed explanations and alternative approaches.

## Exercise 1: Fine‑tune on a different domain (e.g., cooking recipes)

We'll create a dataset of cooking recipes (instruction: recipe name, response: ingredients and steps).

In [ ]:
# Install required packages (if not already)
# !pip install transformers datasets peft accelerate torch

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

# Load base model
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Create a cooking dataset (50 examples – in practice use at least 200)
recipes = [
    {"instruction": "Make pancakes", "response": "Mix flour, eggs, milk. Fry in pan."},
    {"instruction": "Prepare tomato soup", "response": "Sauté onions, add tomatoes, blend."},
    # Add more recipes here or load from a CSV
]

def format_recipe(example):
    return f"Instruction: {example['instruction']}\nResponse: {example['response']}{tokenizer.eos_token}"

texts = [format_recipe(r) for r in recipes]
dataset = Dataset.from_dict({"text": texts})

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],  # For GPT-2
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Should be <1%

# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_cooking",
    per_device_train_batch_size=2,
    num_train_epochs=10,  # More epochs for small dataset
    logging_steps=5,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    report_to="none"  # Disable wandb/tensorboard for simplicity
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

# Train (uncomment to run)
# trainer.train()

# Test generation
prompt = "Instruction: Make pancakes\nResponse:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)
print(tokenizer.decode(outputs[0]))

## Exercise 2: Vary rank (r) and compare

We train three models with r=2, r=8, r=32 and compare trainable parameters and output quality.

In [ ]:
ranks = [2, 8, 32]
results = {}

for r in ranks:
    print(f"\n--- Training with r={r} ---")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    lora_config = LoraConfig(r=r, lora_alpha=4*r, target_modules=["c_attn"], lora_dropout=0.1, task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(model, lora_config)
    trainable = model.num_parameters(only_trainable=True)
    print(f"Trainable parameters: {trainable:,}")
    results[r] = {"trainable": trainable}
    # Train (simplified – in practice you would train each)
    # ... training code ...
    # After training, generate and evaluate
    # For this demo we just store the parameter count

print("\nComparison:")
for r, data in results.items():
    print(f"r={r}: {data['trainable']:,} trainable params")

## Exercise 3: Merge and save the full model

After training, merge LoRA weights into the base model and save.

In [ ]:
# Assuming 'model' is the trained PeftModel
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_gpt2_cooking")
tokenizer.save_pretrained("./merged_gpt2_cooking")
print("Merged model saved to ./merged_gpt2_cooking")

## Exercise 4: Load adapter only onto a fresh base model

This is useful for swapping adapters without duplicating the base model.

In [ ]:
from peft import PeftModel

# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(model_name)
adapter_model = PeftModel.from_pretrained(base_model, "./lora_cooking/checkpoint-xxx")  # replace with actual checkpoint

# Now adapter_model can generate using the fine‑tuned behaviour
inputs = tokenizer("Instruction: Make pancakes\nResponse:", return_tensors="pt")
outputs = adapter_model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0]))

# You can also unload the adapter to revert to base model
base_model2 = adapter_model.unload()
print("Adapter unloaded, back to base model.")

## Additional: Using a larger dataset (optional)

For real fine‑tuning, use at least 500 examples. Load from a CSV or Hugging Face dataset.

In [ ]:
# from datasets import load_dataset
# dataset = load_dataset("csv", data_files="recipes.csv")